In [71]:
# Install dependencies
!pip install pdf2image pillow transformers accelerate torch torchvision sentencepiece timm
!apt-get install poppler-utils -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [72]:
# Imports
import torch
from pdf2image import convert_from_bytes
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    DonutProcessor,
    VisionEncoderDecoderModel
)
from concurrent.futures import ThreadPoolExecutor

In [84]:
# Configuration (from config.py)
MODEL_NAME = "donut_cord"

# options:
# donut_docvqa
# donut_cord
# qwen_vl

In [85]:
# PDF → Images (pdf_utils.py)
def pdf_to_images(pdf_bytes):
    images = convert_from_bytes(pdf_bytes)
    return images

In [75]:
# Markdown Converter (markdown_utils.py)
def convert_to_markdown(pages):

    markdown = ""

    for i, text in enumerate(pages):
        markdown += f"# Page {i+1}\n\n"
        markdown += text + "\n\n"

    return markdown

In [89]:
# Load Models
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)
model_dict = {"MODEL_NAME" : "model_id"}

model_dict = {

    "donut_docvqa" : "naver-clova-ix/donut-base-finetuned-docvqa",

    "donut_cord" : "naver-clova-ix/donut-base-finetuned-cord-v2"

}

if MODEL_NAME in model_dict.keys():
    processor = DonutProcessor.from_pretrained(
        model_dict[MODEL_NAME]
    )

    model = VisionEncoderDecoderModel.from_pretrained(
        model_dict[MODEL_NAME]
    ).to(device)

else:
  raise ValueError("Invalid MODEL_NAME")

Using device: cuda


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [90]:
# Model functions
import torch

def run_qwen(image):

    prompt = """
Read this document page carefully.

Extract ALL visible text exactly as written.
Do NOT describe the image.
Do NOT output coordinates or bounding boxes.

Return only the extracted text in clean Markdown format.
Preserve headings, lists, and tables when possible.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,      # deterministic
            temperature=0.0
        )

    output_text = processor.batch_decode(
        output_ids,
        skip_special_tokens=True
    )[0]

    # Remove chat template artifacts
    if "assistant" in output_text:
        output_text = output_text.split("assistant")[-1]

    return output_text.strip()


def ask_question(image, question):

    task_prompt = f"<s_docvqa>{question}"

    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    decoder_input_ids = processor.tokenizer(
        task_prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=128,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=3
        )

    answer = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    # Remove prompt echo
    answer = answer.replace(task_prompt, "").strip()

    return answer


def run_donut_cord(image):

    task_prompt = "<s_cord-v2>"

    decoder_input_ids = processor.tokenizer(
        task_prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(device)

    pixel_values = processor(
        image,
        return_tensors="pt"
    ).pixel_values.to(device)

    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=1024
    )

    result = processor.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    return result

In [91]:
# Model Router (model_router.py)
def run_model(image):

    if MODEL_NAME == "donut_docvqa":
        return run_donut_docvqa(image)

    elif MODEL_NAME == "donut_cord":
        return run_donut_cord(image)

    elif MODEL_NAME == "qwen_vl":
        return run_qwen(image)

    else:
        raise ValueError("Unknown model selected")

In [80]:
# Upload pdf
from google.colab import files

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

with open(pdf_path, "rb") as f:
    pdf_bytes = f.read()

Saving invoice-0-4.pdf to invoice-0-4 (2).pdf


In [81]:
# Convert PDF → Images
images = pdf_to_images(pdf_bytes)

print("Total pages:", len(images))

Total pages: 2


In [82]:
import torch
import gc
from PIL import Image

total_pages = len(images)
extracted_pages = []

MAX_SIZE = 1400  # reduce image size to control attention memory


def prepare_image(img):
    if max(img.size) > MAX_SIZE:
        img.thumbnail((MAX_SIZE, MAX_SIZE))
    return img


for i, image in enumerate(images):

    print(f"\nProcessing page {i+1}/{total_pages}")

    try:
        image = prepare_image(image)

        with torch.no_grad():
            result = run_model(image)

        extracted_pages.append(result)

    except RuntimeError as e:

        if "out of memory" in str(e):
            print("⚠️ GPU OOM, clearing memory and retrying...")

            torch.cuda.empty_cache()
            gc.collect()

            try:
                with torch.no_grad():
                    result = run_model(image)

                extracted_pages.append(result)

            except Exception as e2:
                print(f"Failed again: {e2}")
                extracted_pages.append("")

        else:
            print(e)
            extracted_pages.append("")

    # HARD MEMORY CLEANUP (important)
    torch.cuda.empty_cache()
    gc.collect()


Processing page 1/2

Processing page 2/2


In [83]:
# Convert to Markdown
markdown = convert_to_markdown(extracted_pages)

print(markdown[:2000])

# Page 1

Extract invoice information

# Page 2

Extract invoice information bpxpn-00070




In [69]:
# Save Result
with open("output.md", "w") as f:
    f.write(markdown)

print("Saved as output.md")

Saved as output.md


In [70]:
# Download Output
from google.colab import files
files.download("output.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>